In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import date
import scanpy as sc
import omicverse as ov
from pathlib import Path
from scipy import sparse

In [ ]:
output_path = Path(r"F:\Data\snRNAseq\PlaceCell\Results\Venn\abslogfc0.1\Expression")
output_path.mkdir(parents=True, exist_ok=True)
today = date.today().strftime("%Y%m%d")

In [ ]:
adata = sc.read_h5ad(r"F:\Data\snRNAseq\PlaceCell\FilteredData\PCAnalysis_all_QC_20260127.h5ad")
adata_annotation = sc.read_h5ad(
    r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData\All_QC_filtered_annotation_20260604.h5ad"
)
assert adata.n_obs == adata_annotation.n_obs
assert (adata.obs_names == adata_annotation.obs_names).all()

# 转移 cell type 注释
adata.obs["cell_types"] = adata_annotation.obs["cell_types"].values

# 去掉 Unassigned
adata = adata[adata.obs["cell_types"] == "CA1-ProS"].copy()

In [ ]:
# Normalize in place
sc.pp.normalize_total(adata, target_sum=1e6)   # A higher target_sum enhances visual contrast

# Log-transform
sc.pp.log1p(adata)

In [ ]:

def export_gene_table_by_condition(
    adata,
    gene: str,
    out_path: str,
    groupby: str = "condition",
    order=("place", "nonplace", "silent"),
    layer: str = None,
    case_insensitive: bool = True,
):
    if gene not in adata.var_names:
        raise ValueError(f"Gene not found: {gene}")
    if groupby not in adata.obs.columns:
        raise ValueError(f"Column not found in obs: {groupby}")

    # 1) expression vector
    x = adata[:, gene].layers[layer] if layer is not None else adata[:, gene].X
    x = x.toarray().ravel() if sparse.issparse(x) else np.asarray(x).ravel()

    # 2) group labels
    g_raw = adata.obs[groupby].astype(str).str.strip()
    if case_insensitive:
        g = g_raw.str.lower()
        keys = [str(k).strip().lower() for k in order]
    else:
        g = g_raw
        keys = [str(k).strip() for k in order]

    # 3) build columns by requested order
    cols = {}
    for out_name, key in zip(order, keys):
        vals = x[g.eq(key).to_numpy()]
        cols[out_name] = pd.Series(vals)

    df = pd.DataFrame(cols)

    # 4) save
    out_dir = os.path.dirname(out_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    df.to_csv(out_path, index=False)

    print("Available groups:", sorted(g.unique().tolist()))
    print("Requested order:", order)
    print("Saved:", out_path, "shape=", df.shape)
    return df


In [ ]:
adata_no_pc118 = adata[adata.obs["sample"].notna() & adata.obs["sample"].ne("PC118")].copy()
adata_no_pc118.obs["condition"].value_counts()

In [ ]:
gene = "Jun"
ov.pl.violin(
    adata,
    keys=gene,
    groupby="condition",
    stripplot=True,
    jitter=True,
    size=2,
)

In [ ]:
import scanpy as sc

sc.pp.calculate_qc_metrics(
    adata,
    layer="counts",
    inplace=True
)

In [ ]:
adata.obs.groupby("condition")["total_counts"].mean()
# adata.obs.groupby("condition")["n_genes_by_counts"].mean()

In [ ]:
out_file = os.path.join(output_path, f"{gene}_by_condition_{today}.csv")
export_gene_table_by_condition(
    adata, gene, out_file,
    groupby="condition",
    order=("place", "nonplace", "silent"),
    layer=None
)
